In [3]:
import os
import json
import torch
import torch.nn as nn
from torch.func import vmap, functional_call, jacrev
import numpy as np

# Set fixed seed for reproducibility across all feature runs
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

class TwoLayerMLPProbe(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int = 256, num_classes: int = 6):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, num_classes)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

def compute_mlp_lgc(features: np.ndarray, labels: np.ndarray, hidden_dim: int = 256, num_classes: int = 6):
    N, D = features.shape
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # 1. Normalize features (L2 sphere)
    norms = np.linalg.norm(features, axis=1, keepdims=True) + 1e-8
    features_norm = torch.tensor(features / norms, dtype=torch.float32, device=device)

    # 2. Instantiate 2-Layer MLP with fixed seed
    torch.manual_seed(SEED)
    model = TwoLayerMLPProbe(input_dim=D, hidden_dim=hidden_dim, num_classes=num_classes).to(device)
    model.eval()

    params = dict(model.named_parameters())
    buffers = dict(model.named_buffers())

    # Single-sample logit function: (D,) -> (C,)
    def predict_single(p, b, x_vec):
        return functional_call(model, (p, b), x_vec.unsqueeze(0)).squeeze(0)

    # Vectorized per-sample Jacobian w.r.t MLP parameters
    sample_jac_fn = vmap(jacrev(predict_single, argnums=0), in_dims=(None, None, 0))
    jac_dict = sample_jac_fn(params, buffers, features_norm)

    # Flatten and concatenate all parameter gradients into (N, C * P_total)
    flat_jacs = [g.reshape(N, -1) for g in jac_dict.values()]
    J = torch.cat(flat_jacs, dim=1)  # Shape: (N, Total_Grad_Dim)

    # Compute empirical NTK: Theta = J @ J.T (N x N)
    Theta = (J @ J.T).detach().cpu().numpy()

    # 3. Kernel Centering
    H = np.eye(N) - (1.0 / N) * np.ones((N, N))
    Theta_tilde = H @ Theta @ H

    Y_onehot = np.eye(num_classes)[labels]
    K_Y = Y_onehot @ Y_onehot.T
    K_Y_tilde = H @ K_Y @ H

    norm_Theta = np.linalg.norm(Theta_tilde, 'fro') + 1e-12
    norm_KY = np.linalg.norm(K_Y_tilde, 'fro') + 1e-12

    # 4. Class-wise and Overall LGC
    class_scores = {}
    total_alignment = 0.0

    for c in range(num_classes):
        y_c = (labels == c).astype(np.float64)
        y_c_tilde = y_c - np.mean(y_c)
        
        # Intra-class gradient correlation
        intra_c = float(y_c_tilde.T @ Theta_tilde @ y_c_tilde)
        score_c = intra_c / (norm_Theta * norm_KY)
        
        class_scores[int(c)] = score_c
        total_alignment += score_c

    return float(total_alignment), class_scores

# --- Pipeline Execution ---
base_dir = "../model_embeddings"
output_dir = "../model_scores/LGC_mlp"
os.makedirs(output_dir, exist_ok=True)

results = {}

for model_name in sorted(os.listdir(base_dir)):
    model_path = os.path.join(base_dir, model_name)
    if not os.path.isdir(model_path):
        continue
        
    npz_path = os.path.join(model_path, "features.npz")
    if not os.path.exists(npz_path):
        continue
        
    print(f"Calculating MLP LGC for {model_name}...")
    data = np.load(npz_path)
    features = data["embeddings"]
    labels = data["labels"]
    
    overall, class_wise = compute_mlp_lgc(features, labels, hidden_dim=256, num_classes=6)
    
    results[model_name] = {
        "overall_lgc": overall,
        "class_wise_lgc": class_wise
    }

output_file = os.path.join(output_dir, "all_models_lgc.json")
with open(output_file, "w") as f:
    json.dump(results, f, indent=4)

print(f"\nSaved MLP NTK-LGC scores to {output_file}")

Calculating MLP LGC for AVES-bio...
Calculating MLP LGC for BirdAVES-biox-base...
Calculating MLP LGC for DistilHuBERT...
Calculating MLP LGC for HuBERT...
Calculating MLP LGC for MusicHuBERT...
Calculating MLP LGC for WavLM...

Saved MLP NTK-LGC scores to ../model_scores/LGC_mlp\all_models_lgc.json
